# 🧠 Aprendizaje No Supervisado de Datos Categóricos
## Análisis de Variables Educativas — EPHPM 2025 (Honduras)
### Restricted Boltzmann Machine · t-SNE · KMeans

---

**Fuente de datos:** Instituto Nacional de Estadística (INE) — Encuesta Permanente de Hogares de Propósitos Múltiples 2025  
**Método:** Inspirado en *Deng, L. (2022). Unsupervised Categorical Learning of Objects and Feature Dimension Reduction Using a Brain-Inspired Algorithm.* Towards Data Science.  
**Unidad de análisis:** Hogar (7,870 hogares)  
**Dominio temático:** Variables educativas (ED02–ED19)

---

### ▶️ Cómo ejecutar este notebook

1. Instala dependencias: `pip install -r requirements.txt`
2. Coloca el CSV de la EPHPM 2025 en `data/EPHPM2025.csv` (los microdatos **no** se incluyen en el repo; ver `data/README.md`). También puedes apuntar a otra ruta con la variable de entorno `EPHPM_CSV`.
3. Sin los microdatos, puedes hacer una *prueba de humo* con datos sintéticos: `python scripts/make_synthetic_data.py` y luego `EPHPM_CSV=data/EPHPM2025_synthetic.csv jupyter lab`. Los resultados sintéticos **no tienen significado sustantivo**; solo comprueban que el pipeline corre.
4. Ejecuta todas las celdas (`Run All`). Las figuras se guardan en `figures/` y el CSV de resultados en `outputs/`.

---

### 📋 Contenido del notebook

| # | Sección | Descripción |
|---|---------|-------------|
| 0 | Configuración | Librerías y parámetros globales |
| 1 | Carga de datos | Exploración inicial del dataset |
| 2 | Feature Engineering | Construcción de variables a nivel hogar |
| 3 | Codificación Binaria | One-hot encoding para el RBM |
| 4 | Restricted Boltzmann Machine | Entrenamiento y reducción de dimensionalidad |
| 5 | Clustering con KMeans | Selección de k óptimo + etiquetas |
| 6 | Visualización t-SNE | Proyecciones 2D de las representaciones aprendidas |
| 7 | Interpretación de resultados | Análisis de los 4 gráficos |
| 8 | Exportación | CSV con etiquetas de cluster |
| 9 | Extensiones propuestas | Próximos pasos y exploraciones |


---
## Sección 0 — Configuración de librerías y parámetros

### 📚 Guía de estudio

Las librerías utilizadas pertenecen al ecosistema científico estándar de Python:

- **`numpy` / `pandas`**: manipulación de matrices y DataFrames
- **`matplotlib` / `seaborn`**: visualización estadística
- **`sklearn.neural_network.BernoulliRBM`**: implementación del RBM con distribución Bernoulli (apropiada para entradas binarias 0/1)
- **`sklearn.manifold.TSNE`**: proyección no lineal para visualización
- **`sklearn.cluster.KMeans`**: agrupamiento por centroides
- **`sklearn.metrics.silhouette_score`**: métrica para seleccionar el número óptimo de clusters

### ⚙️ Parámetros del modelo

Los parámetros clave del RBM son:

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| `N_HIDDEN` | 40 | Neuronas en la capa oculta 1 (dimensión del espacio latente) |
| `N_HIDDEN2` | 20 | Neuronas en la capa oculta 2 (RBM apilado) |
| `N_ITER` | 30 | Épocas de entrenamiento |
| `LR` | 0.01 | Tasa de aprendizaje (qué tanto se actualizan los pesos por iteración) |
| `BATCH` | 64 | Tamaño del mini-lote |


In [ ]:
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.neural_network import BernoulliRBM
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ── Parámetros globales del modelo ────────────────────────────────────────────
N_HIDDEN  = 40    # Unidades ocultas RBM Capa 1
N_HIDDEN2 = 20    # Unidades ocultas RBM Capa 2
N_ITER    = 30    # Épocas de entrenamiento
LR        = 0.01  # Tasa de aprendizaje
BATCH     = 64    # Tamaño del mini-lote
RANDOM    = 42    # Semilla para reproducibilidad

# ── Rutas (configurables) ─────────────────────────────────────────────────────
# ROOT = raíz del repo, tanto si Jupyter se abre en la raíz como en notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
_csv = Path(os.environ.get('EPHPM_CSV', 'data/EPHPM2025.csv'))
DATA_PATH  = _csv if _csv.is_absolute() else ROOT / _csv
FIG_DIR    = ROOT / 'figures';  FIG_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = ROOT / 'outputs';  OUTPUT_DIR.mkdir(exist_ok=True)

def savefig(fig, name):
    """Guarda la figura en figures/ (PNG, 150 dpi) para usarla en el README."""
    fig.savefig(FIG_DIR / f'{name}.png', dpi=150, bbox_inches='tight')

plt.rcParams['figure.dpi'] = 120
print('✓ Librerías cargadas correctamente')


---
## Sección 1 — Carga y exploración inicial de los datos

### 📚 Guía de estudio

La EPHPM (Encuesta Permanente de Hogares de Propósitos Múltiples) es la principal fuente de datos sociodemográficos del INE Honduras. El dataset tiene una estructura **persona-dentro-de-hogar**: cada fila es una persona, y múltiples personas comparten el mismo identificador `HOGAR`.

**Variables clave de estructura:**
- `HOGAR`: identificador único del hogar
- `RELA_J`: relación con el jefe del hogar (1 = jefe/a)
- `NPER`: número de personas en el hogar

**Variables educativas principales:**

| Variable | Descripción |
|----------|-------------|
| `ED02` | ¿Actualmente asiste a algún centro de enseñanza? |
| `ED03` | ¿Alguna vez asistió? |
| `ED08` | Último nivel/grado completado |
| `ED09` | Área o carrera de estudios |
| `ED15` | Sabe leer y escribir |
| `nivel` | Nivel educativo (1=Primaria ... 5=Posgrado, 9=Ninguno) |
| `anosest` | Años de estudio |


In [ ]:
# Carga del dataset
# ── IMPORTANTE ────────────────────────────────────────────────────────────
# El archivo EPHPM2025.csv puede venir con DOS filas de encabezado:
#   fila 0 = nombres cortos de columna (HOGAR, RELA_J, nivel, ...)
#   fila 1 = descripciones largas en español ('Relación con el jefe...', etc.)
# Detectamos automáticamente si existe la fila de descripciones: si el valor de
# HOGAR en la primera fila de datos no es numérico, esa fila es una descripción.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA_PATH}. Coloca el CSV de la EPHPM 2025 en esa ruta, '
        'define EPHPM_CSV, o genera datos sintéticos con scripts/make_synthetic_data.py '
        '(ver data/README.md).'
    )
peek = pd.read_csv(DATA_PATH, nrows=2)
has_desc_row = pd.isna(pd.to_numeric(peek.loc[0, 'HOGAR'], errors='coerce'))
df = pd.read_csv(DATA_PATH, low_memory=False, skiprows=[1] if has_desc_row else None)
print(f'Fila de descripciones detectada y omitida: {has_desc_row}')

# Forzar tipos numéricos en las columnas que usamos en cálculos y comparaciones.
# errors='coerce' convierte cualquier valor no-numérico en NaN.
num_cols = ['RELA_J', 'HOGAR', 'NPER', 'nivel', 'anosest',
            'ED02', 'ED03', 'ED08', 'ED09', 'ED15',
            'SEXO', 'DOMINIO', 'DEPTO']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Dimensiones del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'  (en la corrida original con los microdatos reales: 28,714 filas)')
print(f'Hogares únicos        : {df["HOGAR"].nunique():,}')
print(f'RELA_J dtype          : {df["RELA_J"].dtype}  (debe ser float64)')
print(f'anosest dtype         : {df["anosest"].dtype}  (debe ser float64)')
print(f'Total jefes de hogar  : {(df["RELA_J"]==1).sum():,}  (7,870 en la corrida original)')


In [ ]:
# Exploración de variables educativas clave
print('=== Distribución de nivel educativo ===')
nivel_labels = {1.0:'Primaria', 2.0:'Secundaria', 3.0:'Técnico',
                4.0:'Universidad', 5.0:'Posgrado', 9.0:'Ninguno'}
print(df['nivel'].value_counts().rename(index=nivel_labels))

print('\n=== Estadísticas de años de estudio ===')
print(df['anosest'].clip(0,22).describe().round(2))

print('\n=== Jefes de hogar (RELA_J==1) ===')
print(f'Total jefes de hogar: {(df["RELA_J"]==1).sum():,}')


---
## Sección 2 — Feature Engineering: construcción de variables a nivel hogar

### 📚 Guía de estudio

Como el RBM operará a **nivel hogar** (una fila por hogar), necesitamos transformar el dataset persona-a-persona en uno hogar-a-hogar. Esto implica dos estrategias complementarias:

**1. Variables del jefe del hogar** (`RELA_J == 1`)
> El jefe es una proxy del perfil socioeconómico dominante del hogar. Sus características educativas representan la generación de mayor autoridad/decisión en el núcleo familiar.

**2. Agregados de todos los miembros**
> Capturan la heterogeneidad *intra-hogar*. Un hogar puede tener un jefe con primaria pero un hijo universitario — ese contraste es información estructural valiosa.

| Variable agregada | Cómo se calcula | Qué captura |
|-------------------|-----------------|-------------|
| `hh_max_nivel` | Máximo de `nivel` en el hogar | Capital educativo máximo disponible |
| `hh_mean_anosest` | Promedio de `anosest` | Escolaridad promedio del hogar |
| `hh_has_university` | ¿Hay algún miembro con nivel ≥ 4? | Presencia de educación superior |
| `hh_has_illiterate` | ¿Hay algún miembro con `ED15==1`? ⚠️ | Vulnerabilidad educativa |
| `hh_prop_attending` | Proporción con `ED02==1` | Escolarización activa actual |
| `hh_has_dropout` | ¿Hay algún miembro con `ED03==1`? ⚠️ | Ver nota (¿asistió alguna vez, o nunca asistió?) |


> ⚠️ **Verificar contra el diccionario de variables del INE.** En muchas encuestas de hogares el código `1` significa *“Sí”* y `2` *“No”*. Si ese es el caso aquí, `ED15==1` identificaría a quien **sí** sabe leer y `ED03==1` a quien **sí** asistió alguna vez, lo cual invertiría el significado de `hh_has_illiterate` y de la descripción “nunca asistió”. Antes de interpretar esas dos variables, confirma la codificación exacta en el diccionario de la EPHPM 2025 y, si hace falta, cambia `== 1` por `== 2` en la celda siguiente.


In [ ]:
# ── 2a. Slice del jefe de hogar ──────────────────────────────────────────────
# pd.to_numeric garantiza comparación numérica independiente de cómo se cargó el CSV
rela_j_num = pd.to_numeric(df['RELA_J'], errors='coerce')
head = df[rela_j_num == 1].drop_duplicates(subset='HOGAR', keep='first').copy()
print(f'Jefes de hogar únicos: {len(head):,}')

# ── 2b. Normalización de variables numéricas ──────────────────────────────────
nivel_map = {1.0:1, 2.0:2, 3.0:3, 4.0:4, 5.0:5, 9.0:0}
df['nivel_num'] = df['nivel'].map(nivel_map)
df['anosest_capped'] = df['anosest'].clip(0, 22)  # Cap: 22 años de estudio es el máximo razonable

# ── Tamaño real del hogar ────────────────────────────────────────────────────
# NPER es el número de ORDEN de la persona dentro del hogar (1=jefe, 2=segundo...),
# NO el tamaño del hogar. El tamaño real es el conteo de filas por HOGAR.
hh_size = df.groupby('HOGAR').size().rename('hh_n_members')

# ── 2c. Agregados a nivel hogar ───────────────────────────────────────────────
hh_agg = (
    df.groupby('HOGAR')
    .agg(
        hh_max_nivel     = ('nivel_num',      'max'),
        hh_mean_anosest  = ('anosest_capped',  'mean'),
        hh_has_university= ('nivel_num',       lambda x: int((x >= 4).any())),
        hh_has_illiterate= ('ED15',            lambda x: int((x == 1).any())),
        hh_prop_attending= ('ED02',            lambda x: (x == 1).sum() / len(x)),
        hh_has_dropout   = ('ED03',            lambda x: int((x == 1).any())),
    )
    .reset_index()
)
hh_agg = hh_agg.merge(hh_size, on='HOGAR')  # agregar tamaño real del hogar

# ── 2d. Merge jefe + agregados ────────────────────────────────────────────────
feat_cols_head = ['HOGAR','DOMINIO','DEPTO','SEXO',
                  'ED02','ED03','ED08','ED09','ED15','nivel']
hh = head[feat_cols_head].merge(hh_agg, on='HOGAR', how='inner')
# nivel_num recodifica 9 (Ninguno) → 0 para que el orden sea 0 < 1 < ... < 5.
# Se usa en gráficos y perfiles; el one-hot sigue usando el código original.
hh['nivel_num'] = hh['nivel'].map(nivel_map)
print(f'Dataset a nivel hogar: {len(hh):,} hogares × {len(hh.columns)} variables')


In [ ]:
# ── 2e. Binning de variables continuas → categorías ──────────────────────────
# Necesario para one-hot encoding posterior: el RBM requiere entradas binarias
hh['hh_mean_anosest_bin'] = pd.cut(
    hh['hh_mean_anosest'],
    bins=[-0.1, 3, 6, 9, 12, 22],
    labels=['0-3yr','4-6yr','7-9yr','10-12yr','13+yr']
)
hh['hh_max_nivel_cat'] = hh['hh_max_nivel'].map(
    {0:'Ninguno',1:'Primaria',2:'Secundaria',3:'Técnico',4:'Universidad',5:'Posgrado'}
)
hh['hh_n_members_bin'] = pd.cut(
    hh['hh_n_members'], bins=[0,2,4,6,100],
    labels=['1-2','3-4','5-6','7+']
)

print('Distribución: años de estudio promedio (binned)')
print(hh['hh_mean_anosest_bin'].value_counts().sort_index())


---
## Sección 3 — Codificación Binaria (One-Hot Encoding)

### 📚 Guía de estudio

El `BernoulliRBM` de scikit-learn **requiere entradas en el rango [0, 1]** — idealmente valores exactamente 0 o 1 (binarios). Esto se debe a que modela cada unidad visible como una variable aleatoria Bernoulli:

$$v_i \sim \text{Bernoulli}(\sigma(b_i + \sum_j W_{ij} h_j))$$

Por eso transformamos cada variable categórica en un conjunto de **indicadores binarios**. Para una variable con $k$ categorías, obtenemos $k$ columnas de 0s y 1s.

**Ejemplo:**
```
nivel = 2.0 (Secundaria)
→ Jefe_nivel_educ_1.0 = 0
   Jefe_nivel_educ_2.0 = 1   ← activo
   Jefe_nivel_educ_3.0 = 0
   ...
```

Las columnas con valores `NA` se eliminan (son artefactos de missingness, no información real).


In [ ]:
# ── Definición del mapeo variable original → nombre legible ──────────────────
cat_cols = {
    'DOMINIO':             'Dominio',
    'DEPTO':               'Departamento',
    'SEXO':                'Sexo_jefe',
    'ED02':                'Jefe_asiste_escuela',
    'ED03':                'Jefe_alguna_vez_asistio',
    'ED08':                'Jefe_ultimo_nivel_comp',
    'ED09':                'Jefe_area_estudio',
    'ED15':                'Jefe_sabe_leer',
    'nivel':               'Jefe_nivel_educ',
    'hh_has_university':   'Hogar_tiene_universitario',
    'hh_has_illiterate':   'Hogar_tiene_analfabeto',
    'hh_has_dropout':      'Hogar_tiene_sin_escuela',
    'hh_mean_anosest_bin': 'Hogar_anos_estudio_prom',
    'hh_max_nivel_cat':    'Hogar_max_nivel',
    'hh_n_members_bin':    'Hogar_n_miembros',
}

# ── One-hot encoding ──────────────────────────────────────────────────────────
dummies_list = []
for raw_col, label in cat_cols.items():
    col_data = hh[raw_col].astype(str).fillna('NA')
    dummies  = pd.get_dummies(col_data, prefix=label)
    dummies  = dummies[[c for c in dummies.columns
                         if not c.endswith('_NA') and not c.endswith('_nan')]]
    dummies_list.append(dummies)

X_bin = pd.concat(dummies_list, axis=1).astype(float).fillna(0)

print(f'Matriz binaria final: {X_bin.shape[0]:,} hogares × {X_bin.shape[1]} indicadores')
print('\nIndicadores por grupo:')
for raw_col, label in cat_cols.items():
    n = sum(1 for c in X_bin.columns if c.startswith(label))
    print(f'  {label:<35} {n:>2} indicadores')


---
## Sección 4 — Restricted Boltzmann Machine (RBM)

### 📚 Guía de estudio

#### ¿Qué es un RBM?

Un RBM es una **red neuronal probabilística no dirigida** con dos capas: visible (los datos de entrada) y oculta (representación aprendida). Es *restricted* porque no hay conexiones dentro de la misma capa.

```
CAPA OCULTA:   h₁  h₂  h₃  ...  h₄₀     ← 40 unidades latentes
                ↕   ↕   ↕         ↕       ← pesos W_ij
CAPA VISIBLE:  v₁  v₂  v₃  ...  v₇₂     ← indicadores binarios (72 en la corrida original)
```

#### La función de energía

El RBM define una distribución de probabilidad conjunta a través de una función de energía:

$$E(\mathbf{v}, \mathbf{h}) = -\sum_i b_i v_i - \sum_j c_j h_j - \sum_{i,j} v_i W_{ij} h_j$$

- $b_i$: bias de la unidad visible $i$
- $c_j$: bias de la unidad oculta $j$
- $W_{ij}$: peso de la conexión entre visible $i$ y oculto $j$

**Los pesos $W_{ij}$ son la memoria del modelo**: reflejan qué variables de entrada co-ocurren sistemáticamente.

#### Entrenamiento: Contrastive Divergence

> 🔧 **Detalle de implementación:** `BernoulliRBM` de scikit-learn entrena con *Persistent Contrastive Divergence* (también llamada *Stochastic Maximum Likelihood*), una variante de CD en la que las cadenas de Gibbs “fantasía” persisten entre mini-lotes.

El entrenamiento ajusta los pesos para *minimizar la energía* de los datos reales y *maximizar la energía* de datos generados por el modelo. Intuitivamente: el RBM aprende qué patrones son "normales" en la población.

#### RBM Apilado (Stacked RBM / Deep Belief Network)

Entrenamos una segunda capa RBM sobre las **activaciones ocultas** de la primera capa. Esto permite capturar patrones de orden superior — el equivalente conceptual a capas de abstracción.

```
Datos (72)  →  [RBM 1]  →  H₁ (40)  →  [RBM 2]  →  H₂ (20)
                                                     ↓
                                              KMeans + t-SNE
```

> 💡 **Idea central**: cada hogar, que originalmente era un vector de indicadores binarios (72 en la corrida original), queda representado por un vector de 20 números. Esa representación comprimida resume su perfil educativo.


In [ ]:
# ── Capa 1: RBM (72 → 40 unidades) ──────────────────────────────────────────
print('Entrenando RBM Capa 1 (72 → 40 unidades)...')
rbm = BernoulliRBM(
    n_components  = N_HIDDEN,
    learning_rate = LR,
    n_iter        = N_ITER,
    batch_size    = BATCH,
    random_state  = RANDOM,
    verbose       = False
)
rbm.fit(X_bin.values)
H = rbm.transform(X_bin.values)  # H.shape = (7870, 40)

print(f'Capa 1: {X_bin.shape[1]} → {H.shape[1]} dimensiones  '
      f'({(1 - H.shape[1]/X_bin.shape[1])*100:.0f}% reducción)')

# ── Capa 2: RBM apilado (40 → 20 unidades) ───────────────────────────────────
print('Entrenando RBM Capa 2 (40 → 20 unidades)...')
rbm2 = BernoulliRBM(
    n_components  = N_HIDDEN2,
    learning_rate = LR,
    n_iter        = N_ITER,
    batch_size    = BATCH,
    random_state  = RANDOM,
    verbose       = False
)
rbm2.fit(H)
H2 = rbm2.transform(H)  # H2.shape = (7870, 20)

print(f'Capa 2: {H.shape[1]} → {H2.shape[1]} dimensiones  '
      f'({(1 - H2.shape[1]/X_bin.shape[1])*100:.0f}% reducción total)')
print(f'\nResumen de la representación aprendida:')
print(f'  Cada hogar pasó de {X_bin.shape[1]} variables → {H2.shape[1]} variables latentes')
print(f'  H2 describe el perfil educativo "comprimido" de cada hogar')


In [ ]:
# ── Inspección de los pesos aprendidos (Capa 1) ──────────────────────────────
# rbm.components_ tiene forma (n_hidden, n_visible)
# Cada fila es un 'detector de patrón' aprendido
W = rbm.components_  # (40, 72)

print(f'Forma de la matriz de pesos W: {W.shape}')
print('  → 40 filas = 40 unidades ocultas (patrones latentes)')
print('  → 72 cols  = 72 indicadores binarios del input')
print()
print('Top 5 variables con mayor activación en la Unidad Oculta #1:')
unit1_weights = pd.Series(W[0], index=X_bin.columns).sort_values(ascending=False)
print(unit1_weights.head(5).round(4).to_string())


---
## Sección 5 — Clustering con KMeans

### 📚 Guía de estudio

KMeans agrupa los hogares en $k$ grupos minimizando la distancia intra-cluster. Operamos sobre **H₂** (la representación RBM de 20 dimensiones), no sobre los datos originales — el RBM ya filtró el ruido y codificó las relaciones estructurales.

#### Selección de k: Silhouette Score

El silhouette score mide qué tan bien separados están los clusters:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))}$$

- $a(i)$: distancia promedio del punto $i$ a todos los puntos de su mismo cluster
- $b(i)$: distancia promedio del punto $i$ al cluster más cercano (que no es el suyo)

**Rango:** $[-1, +1]$. Cuanto más cercano a 1, mejor definidos están los clusters.

> 💡 **Procedimiento**: se prueban valores de k entre 2 y 8 y se selecciona el que maximiza el silhouette promedio, es decir, el que produce grupos más compactos y mejor separados según esa métrica.


In [ ]:
# ── Búsqueda del k óptimo ─────────────────────────────────────────────────────
sil_scores = {}
for k in range(2, 9):
    km  = KMeans(n_clusters=k, random_state=RANDOM, n_init=15)
    lbl = km.fit_predict(H2)
    sil = silhouette_score(H2, lbl)
    sil_scores[k] = sil
    print(f'  k={k}  →  puntaje de silueta = {sil:.4f}')

best_k = max(sil_scores, key=sil_scores.get)
print(f'\n★ Mejor k = {best_k}  (puntaje de silueta = {sil_scores[best_k]:.4f})')

# ── Clustering final ─────────────────────────────────────────────────────────
km_final      = KMeans(n_clusters=best_k, random_state=RANDOM, n_init=15)
cluster_labels = km_final.fit_predict(H2)
hh['cluster']  = cluster_labels + 1  # 1-indexado para reportes

print(f'\nTamaño de cada cluster:')
print(hh['cluster'].value_counts().sort_index())


---
## Sección 6 — Visualización con t-SNE

### 📚 Guía de estudio

t-SNE (t-distributed Stochastic Neighbor Embedding) es un algoritmo de **reducción de dimensionalidad para visualización**. Proyecta datos de alta dimensión a 2D preservando las relaciones de vecindad local.

#### Diferencia crítica respecto a KMeans

| | t-SNE | KMeans |
|-|-------|--------|
| **Propósito** | Visualización | Agrupamiento |
| **Salida** | Coordenadas 2D | Etiqueta de cluster |
| **Opera sobre** | H₁ o H₂ | H₂ |
| **Distancias son interpretables?** | ❌ No | ✅ Sí |

> ⚠️ **Importante**: las distancias *absolutas* en un gráfico t-SNE no son interpretables. Lo relevante es si los puntos de un mismo color tienden a quedar juntos, lo que sugiere estructura en los datos.

#### ¿Por qué comparamos tres proyecciones?

- **Sin RBM**: muestra cómo los datos brutos tienen estructura, pero ruidosa
- **Capa 1**: el RBM empezó a separar grupos
- **Capa 2**: la representación más abstracta separa mejor los grupos

Esto es evidencia directa de que el RBM **aprendió algo real** de los datos.


In [ ]:
# ── t-SNE sobre las tres representaciones ────────────────────────────────────
print('Corriendo t-SNE (puede tomar 1-2 minutos)...')

tsne_params = dict(n_components=2, perplexity=40, max_iter=1000,
                   random_state=RANDOM, init='pca')

Z0 = TSNE(**tsne_params).fit_transform(X_bin.values)  # Datos brutos
print('  ✓ t-SNE sobre datos brutos')
Z1 = TSNE(**tsne_params).fit_transform(H)             # RBM Capa 1
print('  ✓ t-SNE sobre RBM Capa 1')
Z2 = TSNE(**tsne_params).fit_transform(H2)            # RBM Capa 2
print('  ✓ t-SNE sobre RBM Capa 2')
print('Listo.')


---
## Sección 7 — Visualización e Interpretación de Resultados

A continuación se generan los 4 gráficos del análisis. Después de cada gráfico hay una sección de interpretación.


In [ ]:
# ── Paletas de colores ────────────────────────────────────────────────────────
NIVEL_LABELS = {0.0:'Ninguno', 1.0:'Primaria', 2.0:'Secundaria',
                3.0:'Técnico', 4.0:'Universidad', 5.0:'Posgrado'}
NIVEL_COLORS = {0.0:'#d62728', 1.0:'#ff7f0e', 2.0:'#2ca02c',
                3.0:'#1f77b4', 4.0:'#9467bd', 5.0:'#17becf'}
# OJO: se usa nivel_num (0=Ninguno ... 5=Posgrado). La columna cruda 'nivel' codifica
# 'Ninguno' como 9, por lo que NIVEL_COLORS/NIVEL_LABELS (claves 0.0–5.0) nunca
# coincidirían con ese grupo y quedaría fuera de los gráficos.
nivel_vals  = hh['nivel_num'].values
cluster_pal = sns.color_palette('tab10', best_k)


### 📊 Figura 1: Comparación t-SNE — Datos brutos vs RBM Capa 1 vs RBM Capa 2


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'EPHPM 2025 – t-SNE coloreado por nivel educativo del jefe del hogar\n'
    'Izq: datos brutos | Centro: RBM Capa 1 | Der: RBM Capa 2',
    fontsize=12, fontweight='bold', y=1.02
)
titles = ['Datos brutos\n(sin RBM, 72 vars)',
          'RBM Capa 1\n(40 unidades ocultas)',
          'RBM Capa 2 apilada\n(20 unidades ocultas)']
Zs = [Z0, Z1, Z2]

for ax, Z, title in zip(axes, Zs, titles):
    for nv, color in NIVEL_COLORS.items():
        mask = nivel_vals == nv
        if mask.sum() == 0: continue
        ax.scatter(Z[mask,0], Z[mask,1], c=color, s=6, alpha=0.55,
                   label=NIVEL_LABELS[nv], rasterized=True)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')

handles = [mpatches.Patch(color=NIVEL_COLORS[nv], label=NIVEL_LABELS[nv])
           for nv in sorted(NIVEL_COLORS)]
fig.legend(handles=handles, title='Nivel educativo', loc='lower center',
           ncol=6, fontsize=9, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
savefig(fig, 'fig1_tsne_comparacion')
plt.show()


#### 🔍 Interpretación — Figura 1

Este gráfico muestra la **progresiva separación de los grupos educativos** a medida que los datos pasan por las capas del RBM.

- **Panel izquierdo (datos brutos):** Los puntos están relativamente mezclados. Hay alguna estructura, pero los grupos educativos se solapan considerablemente. El ruido en las 72 variables dificulta la separación.

- **Panel central (RBM Capa 1):** Los grupos empiezan a separarse visualmente. Los hogares con *Ninguno/Primaria* (rojo/naranja) comienzan a agruparse en regiones distintas a los hogares con *Universidad/Posgrado* (morado/cyan).

- **Panel derecho (RBM Capa 2):** La separación es la más nítida. Esto es consistente con que la segunda capa RBM haya aprendido una **representación más abstracta** del perfil educativo de los hogares.

> ⚠️ **Cautela:** t-SNE es una herramienta de *inspección visual*; una mejor separación aparente no prueba por sí sola que el RBM aporte algo. Para sostener esa afirmación hace falta (a) una línea base (p. ej. PCA/SVD sobre los mismos 0/1) y (b) una métrica cuantitativa (p. ej. *trustworthiness* o la precisión de un clasificador sencillo sobre `nivel`). Ver `docs/EXTENDING.md`, extensión 1.

> **Lectura:** sin recibir ninguna etiqueta como objetivo, el RBM produjo una representación en la que los hogares se organizan de forma coherente con el nivel educativo. Como `nivel` forma parte de los datos de entrada, esto es una comprobación de consistencia interna y no una validación externa (ver Limitaciones en el README).


### 📊 Figura 2: Clusters KMeans sobre representación RBM Capa 2


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(f'KMeans Clusters (k={best_k}) — EPHPM 2025',
             fontsize=13, fontweight='bold')

# Izquierda: clusters
ax = axes[0]
for k in range(best_k):
    mask = cluster_labels == k
    ax.scatter(Z2[mask,0], Z2[mask,1], c=[cluster_pal[k]], s=8, alpha=0.6,
               label=f'Cluster {k+1}', rasterized=True)
ax.set_title('Clusters KMeans', fontsize=11, fontweight='bold')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.legend(markerscale=2, fontsize=9)

# Derecha: nivel educativo
ax = axes[1]
for nv, color in NIVEL_COLORS.items():
    mask = nivel_vals == nv
    if mask.sum() == 0: continue
    ax.scatter(Z2[mask,0], Z2[mask,1], c=color, s=8, alpha=0.55,
               label=NIVEL_LABELS[nv], rasterized=True)
ax.set_title('Nivel educativo del jefe del hogar', fontsize=11, fontweight='bold')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.legend(markerscale=2, fontsize=9)

plt.tight_layout()
savefig(fig, 'fig2_clusters_kmeans')
plt.show()


#### 🔍 Interpretación — Figura 2

Este gráfico tiene dos paneles que deben leerse en conjunto:

- **Panel izquierdo (clusters KMeans):** Muestra los 4 grupos identificados automáticamente por el algoritmo. Cada color es un cluster. La separación visual confirma que KMeans encontró grupos geométricamente bien definidos en el espacio RBM.

- **Panel derecho (nivel educativo):** El mismo espacio t-SNE, coloreado por `nivel` — uno de los indicadores que sí alimentó al RBM. Esto es una prueba de consistencia interna (¿el espacio aprendido sigue organizado por nivel?), no una validación externa independiente.

**La coincidencia entre ambos paneles es informativa, con un matiz:** los clusters de KMeans se corresponden con gradientes educativos en parte porque `nivel` fue uno de los inputs del RBM. La validación más fuerte vendría de cruzar los clusters con variables que *nunca* tocaron el modelo, como `Pobreza` o `nbi` (ver Sección 9).

> **Pregunta abierta:** ¿qué proporción de hogares de cada cluster recibe transferencias sociales (p. ej. Bono Vida Mejor)? Cruzar los clusters con variables de política es un siguiente paso directo.


### 📊 Figura 3: Heatmap de perfiles de cluster


In [ ]:
profile_vars = {
    'nivel_num':          'Nivel educ. jefe (media)',
    'hh_max_nivel':       'Nivel máx. hogar (media)',
    'hh_mean_anosest':    'Años estudio prom. hogar',
    'hh_has_university':  'Tiene universitario (%)',
    'hh_has_illiterate':  'Tiene analfabeto (%)',
    'hh_prop_attending':  'Prop. asistiendo escuela',
    'hh_n_members':       'Nº miembros',
}
profile_df = (
    hh.groupby('cluster')[list(profile_vars.keys())]
    .mean()
    .rename(columns=profile_vars)
)
profile_norm = (profile_df - profile_df.min()) / (profile_df.max() - profile_df.min() + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                          gridspec_kw={'width_ratios': [2, 1]})
fig.suptitle('Perfil de Clusters – Variables Educativas (EPHPM 2025)',
             fontsize=13, fontweight='bold')

sns.heatmap(profile_norm.T, ax=axes[0],
            annot=profile_df.T.round(2), fmt='g',
            cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label':'Valor normalizado'},
            annot_kws={'size':9})
axes[0].set_title('Características medias (valores reales anotados)', fontsize=10)
axes[0].set_xlabel('Cluster')

sizes = hh['cluster'].value_counts().sort_index()
bars  = axes[1].bar(sizes.index, sizes.values,
                    color=[cluster_pal[i] for i in range(best_k)],
                    edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, sizes.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
                 f'{val:,}', ha='center', fontsize=10)
axes[1].set_title('Tamaño de cada cluster', fontsize=10)
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Nº hogares')
axes[1].set_xticks(sizes.index)

plt.tight_layout()
savefig(fig, 'fig3_perfiles_cluster')
plt.show()

print('\nValores exactos por cluster:')
print(profile_df.round(3).to_string())


#### 🔍 Interpretación — Figura 3

Este es el gráfico que mejor resume los resultados. Cada celda del heatmap muestra el valor promedio de una variable (fila) dentro de cada cluster (columna). El color refleja el valor normalizado: **amarillo=bajo, rojo=alto**.

> 📝 **Nota de reproducibilidad:** la tabla siguiente interpreta la corrida original. Este notebook ahora promedia `nivel_num` (con *Ninguno* = 0) en lugar del código crudo `nivel` (donde *Ninguno* = 9 inflaba la media del jefe en clusters con muchos jefes sin escolaridad); revisa que cada descripción siga coincidiendo con tu heatmap.

**Perfiles emergentes (leer columna por columna):**

| Cluster | Perfil | Implicación de política |
|---------|--------|------------------------|
| **1** | Alto capital educativo: nivel máximo del hogar elevado, tiene universitarios, pocos analfabetos | Hogares con mayor capacidad para insertarse en mercados laborales formales |
| **2** | Vulnerabilidad extrema: alta presencia de analfabetos, bajo nivel del jefe, proporción alta de miembros en edad escolar | Prioridad para programas de alfabetización y transferencias condicionadas |
| **3** | Heterogeneidad interna: coexisten universitarios y analfabetos en el mismo hogar | Hogares multigeneracionales; la brecha educativa es intra-hogar, no solo inter-hogar |
| **4** | Secundaria moderada sin vulnerabilidad extrema, baja asistencia actual | Posiblemente hogares adultos donde los hijos ya terminaron el ciclo escolar |

> **El panel derecho** muestra que los clusters tienen tamaños distintos. El cluster más grande no necesariamente es el más urgente — es el que cruza con más hogares, pero la intensidad de la necesidad varía.


### 📊 Figura 4: Curva de Silhouette — Selección de k


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ks   = list(sil_scores.keys())
sils = [sil_scores[k] for k in ks]
ax.plot(ks, sils, 'o-', color='#1f77b4', linewidth=2, markersize=8)
ax.axvline(best_k, color='red', linestyle='--', alpha=0.7,
           label=f'Óptimo k={best_k}')
ax.set_xlabel('Número de clusters (k)', fontsize=11)
ax.set_ylabel('Puntaje de Silueta', fontsize=11)
ax.set_title('Selección de k óptimo — Puntaje de Silueta sobre RBM Capa 2',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
savefig(fig, 'fig4_silhouette')
plt.show()


#### 🔍 Interpretación — Figura 4

Esta figura documenta el **proceso de selección del número de clusters**. Cada punto es el silhouette score promedio para ese valor de k.

- **El pico (línea roja)** indica el k donde los clusters son más compactos y mejor separados entre sí.
- Un score cercano a 1.0 indica clusters muy bien definidos; cercano a 0 indica clusters solapados.
- La caída después del pico muestra que agregar más clusters empieza a fragmentar grupos naturales.

> **Nota:** esta gráfica responde la pregunta *"¿por qué 4 clusters y no 3 o 5?"* La respuesta es cuantitativa: en la corrida original k=4 maximizó el silhouette (verifica el pico en tu figura). Además, el silhouette suele favorecer pocos clusters y no mide si los grupos son *útiles* para política pública; conviene contrastarlo con otros criterios (ver `docs/EXTENDING.md`).


---
## Sección 8 — Exportación de resultados

El CSV exportado (en `outputs/`, ignorado por git porque deriva de microdatos del INE) contiene, para cada hogar, sus variables originales clave, la etiqueta de cluster asignada, y las coordenadas t-SNE para reproducir la visualización sin volver a correr el modelo.


In [ ]:
output_df = hh[['HOGAR','DEPTO','DOMINIO','SEXO','nivel',
                 'hh_max_nivel','hh_mean_anosest','hh_n_members',
                 'hh_has_university','hh_has_illiterate','cluster']].copy()
output_df['tsne1_rbm2'] = Z2[:, 0]
output_df['tsne2_rbm2'] = Z2[:, 1]

out_path = OUTPUT_DIR / 'ephpm2025_cluster_labels.csv'
output_df.to_csv(out_path, index=False)
print(f'✓ Archivo exportado: {out_path}')
print(f'  Filas    : {len(output_df):,}')
print(f'  Columnas : {len(output_df.columns)}')
print()
print('Muestra:')
output_df.head()


---
## Sección 9 — Extensiones y próximas exploraciones

El análisis actual es una base sólida. A continuación se presentan extensiones ordenadas por **impacto × esfuerzo**.

---

### 🔴 Alta prioridad — Validez de los resultados

#### 1. Incorporar factores de expansión (`FACTOR`)
La EPHPM es una encuesta por muestreo complejo. Cada hogar tiene un peso que representa a más hogares en la población. Sin incorporar el `FACTOR`, el análisis describe la *muestra*, no Honduras.

```python
# Estrategia: usar FACTOR como peso en el KMeans o en el análisis post-cluster
from sklearn.utils import check_array

# Opción A: ponderar la visualización por tamaño del marcador
factor_jefe = head.set_index('HOGAR')['FACTOR'].reindex(hh['HOGAR']).values
ax.scatter(Z2[:,0], Z2[:,1], s=factor_jefe/factor_jefe.max()*20, ...)

# Opción B: calcular estadísticas ponderadas por cluster
hh['factor'] = factor_jefe
hh.groupby('cluster').apply(lambda g: np.average(g['nivel'], weights=g['factor']))
```

#### 2. Comparación temporal (EPHPM 2024 vs 2025)
Correr el mismo pipeline sobre dos ediciones y comparar la distribución de clusters.

```python
# Correr pipeline idéntico en df_2024 → cluster_2024
# Correr pipeline idéntico en df_2025 → cluster_2025
# Comparar: ¿el Cluster 2 (alta vulnerabilidad) creció o se redujo?
```

---

### 🟡 Prioridad media — Profundización analítica

#### 3. Análisis multi-bloque (educación + vivienda + laboral + TIC)
Incorporar variables de otros bloques temáticos para capturar la estructura socioeconómica completa del hogar.

```python
# Agregar variables de vivienda (material paredes, piso, techo)
# Agregar acceso a servicios (agua, saneamiento, electricidad)
# Agregar variables de TIC (internet, computadora, smartphone)
# Resultado: clusters que reflejan privación multidimensional
```

#### 4. Validación externa con NBI y Pobreza
Los clusters NO fueron entrenados con `Pobreza` ni `nbi`. Cruzarlos es una prueba de validez:

```python
validacion = hh.merge(df[['HOGAR','Pobreza','nbi']].drop_duplicates('HOGAR'),
                      on='HOGAR', how='left')
pd.crosstab(validacion['cluster'], validacion['Pobreza'], normalize='index')
# Si Cluster 2 tiene 80%+ de pobreza extrema → validación fuerte
```

#### 5. Análisis geográfico por departamento
```python
# ¿Qué departamentos concentran al Cluster 2 (alta vulnerabilidad)?
geo = hh.groupby(['DEPTO','cluster']).size().unstack(fill_value=0)
geo_norm = geo.div(geo.sum(axis=1), axis=0)  # proporción por depto
sns.heatmap(geo_norm, cmap='YlOrRd')
```

---

### 🟢 Exploración avanzada 

#### 6. Autoencoder variacional (VAE) como alternativa al RBM
Los autoencoders modernos superan al RBM en capacidad de representación. El VAE además permite *generar* hogares sintéticos.

```python
import torch
import torch.nn as nn

class VAE(nn.Module):
    def __init__(self, input_dim=72, latent_dim=20):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, latent_dim * 2)  # mu + log_var
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, input_dim), nn.Sigmoid()
        )
    # ... reparameterization trick + ELBO loss
```

#### 7. Cramér's V para medir asociación entre clusters y variables externas
```python
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    ct  = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n    = ct.sum().sum()
    return np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

# Requiere el DataFrame `validacion` de la extensión 4 (cluster + Pobreza + nbi)
# Asociación cluster ↔ DEPTO, cluster ↔ Pobreza, cluster ↔ NBI
for var in ['DEPTO','Pobreza','nbi','DOMINIO']:
    v = cramers_v(validacion['cluster'].astype(str), validacion[var].astype(str))
    print(f'  cluster ↔ {var}: V = {v:.3f}')
```

#### 8. Deep Belief Network (DBN) con 3 capas RBM
Agregar una tercera capa RBM (10 unidades) para mayor abstracción. Referencia: Goodfellow et al., Capítulo 20.

```python
rbm3 = BernoulliRBM(n_components=10, learning_rate=0.005,
                    n_iter=50, random_state=RANDOM)
rbm3.fit(H2)
H3 = rbm3.transform(H2)  # 72 → 40 → 20 → 10
```

---

### 📖 Referencias para profundizar

| Recurso | Para qué |
|---------|----------|
| Goodfellow et al. *Deep Learning*, Cap. 20 | Teoría formal del RBM y DBN |
| Larochelle, H. — Video series on RBMs | Explicación visual de Contrastive Divergence |
| distill.pub — How to Use t-SNE Effectively | Interpretación correcta de t-SNE |
| sklearn docs — BernoulliRBM | Parámetros y limitaciones de la implementación |
| INE Honduras — Metodología EPHPM | Marco muestral y factores de expansión |


> 📂 Guía detallada, con orden sugerido y esqueletos de código: [`docs/EXTENDING.md`](../docs/EXTENDING.md)

